# 第 2 章 — 遷移状態探索 (HCN ⇌ HNC, xTB)

**ゴール**
- xTB calculator を ASE 経由で使えるようになる
- 反応物・生成物を最小化して、TS 初期推定構造を **再現可能な関数** で組み立てる
- `Sella(order=1)` で 1 次鞍点を見つけ、**振動解析で虚振動 1 個** を確認する

**題材**: HCN ⇌ HNC の異性化。H が C–N 軸の片側からもう片側へ、弧を描いて移る教科書的な 1 次鞍点です。

## なぜ Sella なのか — BFGS との違い (おさらい)

1 章で見たように、**最小化** は全方向で力をゼロにする点を探します。BFGS は近似 Hessian を使って下り方向に進むだけなので、エネルギーが「上がる方向」には自発的に進みません。

ところが TS (1 次鞍点) は、ある 1 方向 (反応座標) には **エネルギーが極大**、他の方向には極小という鞍の形をしています。BFGS では絶対に登れません。

Sella は

1. Hessian の **部分対角化** で「最も負の固有値の方向」を見つけ、
2. その方向には **登り**、他の方向には **下る**

という更新を行うことで、鞍点に収束させます。`order=1` のときは「負固有値ちょうど 1 個」を狙い、`order=0` のときは「負固有値ゼロ (普通の最小化)」を狙う、と覚えれば OK です。

## 事前準備

```bash
pip install tblite           # Linux/macOS
# Windows や conda 環境: conda install -c conda-forge tblite-python
```

In [ ]:
import numpy as np
from ase import Atoms
from tblite.ase import TBLite
from sella import Sella

def xtb():
    return TBLite(method='GFN2-xTB', verbosity=0)


def hcn_geometry(angle_deg, r_cn=1.17, r_ch=1.10):
    """H-C-N の屈曲角を指定して HCN を組む。
    angle_deg = 180° で線形 HCN、0° で線形 HNC (H が N 側) になる。
    C を原点、N を x 軸正方向に置く。
    """
    # 屈曲角を H-C-N の内角 (degree) として解釈
    theta = np.deg2rad(180.0 - angle_deg)
    H = [-r_ch * np.cos(theta), r_ch * np.sin(theta), 0.0]
    return Atoms('HCN', positions=[H, [0.0, 0.0, 0.0], [r_cn, 0.0, 0.0]])


# 反応物: HCN は angle=180° (線形)
hcn = hcn_geometry(180.0)
hcn.calc = xtb()
Sella(hcn, order=0, logfile=None).run(fmax=1e-3, steps=200)
E_hcn_xtb = hcn.get_potential_energy()
print(f'HCN  最適化後エネルギー: {E_hcn_xtb:.5f} eV')

In [ ]:
# 生成物: HNC は angle=0° (線形, H が N 側)
hnc = hcn_geometry(0.0)
hnc.calc = xtb()
Sella(hnc, order=0, logfile=None).run(fmax=1e-3, steps=200)
E_hnc_xtb = hnc.get_potential_energy()
print(f'HNC  最適化後エネルギー: {E_hnc_xtb:.5f} eV')
print(f'ΔE (HNC - HCN) = {(E_hnc_xtb - E_hcn_xtb)*1000:.1f} meV')

## TS 推定構造を組み立てる

HCN ⇌ HNC の TS は、**H が C と N の間を弧を描いて移る** 折れ曲がった構造です。座標を直接書くと「呪文」に見えるので、上で定義した `hcn_geometry(angle)` を使い、屈曲角を **90° (C-N に対して垂直)** に置いた構造を初期推定とします。

これで「TS は HCN 側 (180°) と HNC 側 (0°) のちょうど中間付近」という直感が幾何にそのまま乗ります。

In [ ]:
ts_guess = hcn_geometry(angle_deg=90.0)
ts_guess.calc = xtb()
print(f'初期 H 座標 : {ts_guess.positions[0]}')
print(f'初期エネルギー: {ts_guess.get_potential_energy():.5f} eV')

## Sella(order=1) で鞍点を最適化

In [ ]:
opt = Sella(
    ts_guess,
    order=1,
    trajectory='ts.traj',
    logfile='ts.log',
)
opt.run(fmax=1e-3, steps=500)

E_ts_xtb = ts_guess.get_potential_energy()
print(f'TS エネルギー : {E_ts_xtb:.5f} eV')
print(f'順方向障壁 Ea(HCN→HNC) = {(E_ts_xtb - E_hcn_xtb):.3f} eV')
print(f'逆方向障壁 Ea(HNC→HCN) = {(E_ts_xtb - E_hnc_xtb):.3f} eV')

## 振動解析で TS を検証する

真の 1 次鞍点なら、**Hessian の固有値のうちちょうど 1 つが負** になり、振動解析の出力には「虚振動」が 1 つ現れます。

**並進回転モードについて**: 非直線分子は 6 個 (並進 3 + 回転 3)、直線分子は 5 個の自由度がほぼゼロ振動として残ります。HCN-HNC の TS は非直線なので 6 個のゼロ振動 + (3N-6 = 3) 個の真の振動 + その中の 1 個が虚、という構成になります。

**ASE の戻り値について**: `Vibrations.get_frequencies()` の戻り値は cm⁻¹ 単位の `np.ndarray` ですが、**虚振動は版によって `負の実数` または `-i*x` の複素数として返ります**。両対応で判定します。

In [ ]:
import shutil, os
from ase.vibrations import Vibrations

# Vibrations は中間ファイルを 'vib/' に書き出す。再実行のため毎回消しておく
if os.path.isdir('vib'):
    shutil.rmtree('vib')

vib = Vibrations(ts_guess, name='vib/ts')
vib.run()
vib.summary()

freqs = vib.get_frequencies()   # 並進回転 + 振動、cm^-1

def is_imaginary(f, thresh=1.0):
    """虚振動判定: 複素表記 (imag != 0) または 負の実数表記 の両方に対応。"""
    if np.iscomplexobj(f):
        return abs(f.imag) > thresh
    return f < -thresh        # 一部の ASE 版は負の実数で返す

n_imag = int(sum(1 for f in freqs if is_imaginary(f)))
print(f'\n虚振動の数: {n_imag} (1 であれば 1 次鞍点)')
print(f'(参考) 全モード数: {len(freqs)} = 並進回転 6 + 振動 {len(freqs)-6}')

## 後続の章にエネルギーを引き継ぐ

5 章 (MACE-MP-0 との比較) では、ここで得た xTB 値を再利用します。IPython の `%store` マジックで保存しておきます。

In [ ]:
%store E_hcn_xtb E_hnc_xtb E_ts_xtb

## 演習

1. `hcn_geometry(angle_deg=120)` や `hcn_geometry(60)` を初期推定にして再実行してください。同じ TS に収束する角度の範囲を探ると、「初期推定の鞍点引力域」の感覚が掴めます。
2. `order=2` で実行したとき、どんな構造に落ちるか確認しましょう。**ヒント**: 振動解析を流し、虚振動が 2 個出ているか確認してください。化学的には意味のない構造ですが、Sella の挙動として勉強になります。
3. `method='GFN1-xTB'` に変えるとエネルギーや障壁高さがどれくらい変わるか比較してください。

---
次章ではこの TS から IRC を流して、本当に HCN ⇄ HNC を繋いでいるかを確かめます。